<a href="https://colab.research.google.com/github/genaiconference/Agentic_KAG_Workshop_DHS_2026/blob/main/05_graph_communities_msgraphrag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GraphRAG Communities on Neo4j — Concept & Walkthrough

## What is this notebook?
It takes an **existing Neo4j knowledge graph** and enriches it with **communities**:
clusters of closely-related entities, each with an LLM-generated summary. It then
exposes those summaries as **retrieval tools for an agent**. Everything is done with the
open-source [`ms-graphrag-neo4j`](https://github.com/neo4j-contrib/ms-graphrag-neo4j)
library, which implements Microsoft's **GraphRAG** methodology.

## Why communities? (the core idea)
Classic RAG retrieves a handful of text chunks — great for **local** questions
(*"What is entity X?"*) but poor for **global** questions
(*"What are the main themes across everything?"*), because that answer is spread across
the whole corpus and lives in no single chunk.

GraphRAG solves this with **Query-Focused Summarization (QFS)**:
1. Build a graph of **entities** and **relationships**.
2. Use the **Leiden** algorithm (via Neo4j GDS) to detect **communities** — hierarchical
   clusters of related entities.
3. Have an LLM write a **summary report** for each community.
4. Answer global questions by map-reducing over these community reports instead of raw text.

## The pipeline in this notebook
| Step | Section | Library call |
|------|---------|--------------|
| Connect + init | 1 | `MsGraphRAG(driver, model)` |
| Inspect graph | 2 | `ms_graph.query(...)` |
| (Optional) schema mapping | 3 | plain Cypher |
| (Optional) summarize nodes/rels | 3b | `summarize_nodes_and_rels()` |
| **Detect + summarize communities** | 4 | `summarize_communities()` |
| Inspect results | 5 | `ms_graph.query(...)` |
| Build agent retrieval tools | 6 | custom `global_` / `local_` search |

## Key schema (what the library reads/writes)
- **`__Entity__`** — the nodes to cluster.
- **`RELATIONSHIP`** — edges Leiden uses for **community detection**.
- **`SUMMARIZED_RELATIONSHIP`** + entity `summary` — used to write **community reports**.
- **`__Community__`** — output nodes, each with `title`, `summary`, `rating`, `level`,
  linked to members via `IN_COMMUNITY`.

## Prerequisites
- Neo4j **5.26+** with the **APOC** and **GDS** plugins enabled.
- An `OPENAI_API_KEY` (and Neo4j credentials) in a local `.env` file.
- An existing graph. If it wasn't built by this library, run the optional cells in
  sections 3 / 3b to align it to the schema above.

---

# Community Detection on an Existing Neo4j Graph — using `ms-graphrag-neo4j`

Instead of hand-writing GDS/Cypher, we call the library's `MsGraphRAG` class and its
**`summarize_communities()`** method, which internally:

1. Runs the **Leiden** algorithm via Neo4j **GDS** to detect communities.
2. Builds the hierarchical `__Community__` node structure.
3. Uses an **OpenAI** model to generate a **Query-Focused Summary (QFS)** report for each community.

**What we do here**
- Connect to Neo4j with credentials from `.env`.
- Initialise `MsGraphRAG` on the *existing* graph.
- Call `summarize_communities()` to detect + summarize communities.
- Inspect the resulting `__Community__` nodes and expose them as agent tools.

> **Requirements:** Neo4j 5.26+, **APOC** + **GDS** plugins, and an `OPENAI_API_KEY`.
> The library expects entities to use the `__Entity__` label. **Community *detection*
> (Leiden) runs on `RELATIONSHIP` edges**, while the LLM community *reports* use
> `SUMMARIZED_RELATIONSHIP` edges + entity `summary` properties (both produced by
> `summarize_nodes_and_rels()`). If your existing graph uses different labels, see the
> mapping cell in **section 3**.

In [ ]:
!git clone https://github.com/genaiconference/Agentic_KAG_Workshop_DHS_2026.git

In [ ]:
# Install the actual MsGraphRAG-Neo4j library (run once)
# The package is distributed via GitHub, so we install straight from the repo.
%pip install -q git+https://github.com/neo4j-contrib/ms-graphrag-neo4j.git
%pip install -q python-dotenv

## 1. Connect to the existing Neo4j graph

Credentials are read from the `.env` file in this folder.

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
from ms_graphrag_neo4j import MsGraphRAG

load_dotenv()  # loads variables from .env in this folder

# MsGraphRAG reads these from the environment
NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

# The library requires OPENAI_API_KEY to be set in the environment
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

# Standard Neo4j driver against the EXISTING graph
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)
driver.verify_connectivity()
print(f"✅ Connected to Neo4j at {NEO4J_URI} (database: {NEO4J_DATABASE})")

# Initialise the MsGraphRAG library on the existing graph.
# create_constraints=True will (idempotently) ensure the __Entity__/__Community__
# constraints the library relies on. It also verifies APOC + GDS are installed.
ms_graph = MsGraphRAG(
    driver=driver,
    model="gpt-5-mini",
    database=NEO4J_DATABASE,
    create_constraints=True,
)
print("✅ MsGraphRAG initialised (APOC + GDS verified by the library).")

## 2. Inspect the existing graph

The library already verified **APOC** and **GDS** during initialisation (it raises an
error otherwise). Here we just look at the labels and relationship types in the graph,
using the library's own `ms_graph.query(...)` helper.

In [ ]:
# Use the library's own query() method (same driver / database under the hood)
print("Node labels:")
for r in ms_graph.query("CALL db.labels() YIELD label RETURN label ORDER BY label"):
    print("  -", r["label"])

print("\nRelationship types:")
for r in ms_graph.query(
    "CALL db.relationshipTypes() YIELD relationshipType AS t RETURN t ORDER BY t"
):
    print("  -", r["t"])

# How many nodes use the library's expected __Entity__ schema?
entity_counts = ms_graph.query(
    """
    MATCH (n:__Entity__)
    OPTIONAL MATCH (n)-[r:RELATIONSHIP]-()
    OPTIONAL MATCH (n)-[sr:SUMMARIZED_RELATIONSHIP]-()
    RETURN count(DISTINCT n) AS entities,
           count(DISTINCT r) AS relationship_edges,
           count(DISTINCT sr) AS summarized_edges
    """
)[0]
print(f"\n__Entity__ nodes: {entity_counts['entities']}")
print(f"RELATIONSHIP edges (used by Leiden detection): {entity_counts['relationship_edges']}")
print(f"SUMMARIZED_RELATIONSHIP edges (used by report step): {entity_counts['summarized_edges']}")

## 3. (Only if needed) Map your existing schema to the library's schema

`ms-graphrag-neo4j` operates on nodes labelled **`__Entity__`**. Community *detection*
(Leiden) is projected on **`RELATIONSHIP`** edges between entities (see the library's
`create_gds_graph_query`), and the community *report* step additionally reads
**`SUMMARIZED_RELATIONSHIP`** edges (created by `summarize_nodes_and_rels()`, section 3b).

If your existing graph already uses those, **skip this cell**. Otherwise, run the
cell below to add the `__Entity__` label and mirror your relationships into
**`RELATIONSHIP`** so Leiden has edges to work with. **Edit the two
variables to match your graph.**

In [ ]:
# ⚠️ OPTIONAL — only run if your graph does NOT already use the library's schema.
# Set these to the label / relationship type your existing graph uses.
RUN_SCHEMA_MAPPING = False          # flip to True to execute the mapping

YOUR_ENTITY_LABEL = "Entity"        # <-- your existing entity label
YOUR_REL_TYPE     = "RELATED"       # <-- your existing relationship type

if RUN_SCHEMA_MAPPING:
    # 1) add the __Entity__ label the library expects
    ms_graph.query(
        f"""
        MATCH (n:`{YOUR_ENTITY_LABEL}`)
        SET n:__Entity__
        """
    )
    # 2) mirror your relationships into RELATIONSHIP (used by Leiden detection).
    #    summarize_nodes_and_rels() (section 3b) will later derive
    #    SUMMARIZED_RELATIONSHIP from these for the report step.
    ms_graph.query(
        f"""
        MATCH (a:__Entity__)-[r:`{YOUR_REL_TYPE}`]->(b:__Entity__)
        MERGE (a)-[nr:RELATIONSHIP]->(b)
        SET nr.description = coalesce(nr.description, 'related')
        """
    )
    print("✅ Schema mapped to __Entity__ / RELATIONSHIP")
else:
    print("Skipped schema mapping (RUN_SCHEMA_MAPPING = False).")

## 3b. (Recommended if graph wasn't built by this library) Summarize nodes & relationships

`summarize_communities()` writes richer community reports when each entity/relationship
has a `summary` property, and it reads relationships of type `SUMMARIZED_RELATIONSHIP`.
The library's own `summarize_nodes_and_rels()` produces exactly these:

- it summarizes each `__Entity__` into `e.summary`,
- and mirrors `RELATIONSHIP` edges into `SUMMARIZED_RELATIONSHIP` with an `r.summary`.

If your existing graph was **created by this library**, it already has them — skip this.
Otherwise, run the cell below (it uses LLM calls, so it costs tokens). It requires your
entities to carry a `description` list property; adjust if your schema differs.

In [ ]:
# ⚠️ OPTIONAL & LLM-costly — creates e.summary + SUMMARIZED_RELATIONSHIP edges.
RUN_NODE_REL_SUMMARIZATION = False   # flip to True if your graph lacks summaries

if RUN_NODE_REL_SUMMARIZATION:
    # async method -> await it
    result = await ms_graph.summarize_nodes_and_rels()
    print(result)
else:
    print("Skipped node/rel summarization (RUN_NODE_REL_SUMMARIZATION = False).")

# Quick check: do entities have summaries and SUMMARIZED_RELATIONSHIP edges?
check = ms_graph.query(
    """
    MATCH (e:__Entity__)
    WITH count(e) AS entities, count(e.summary) AS with_summary
    OPTIONAL MATCH ()-[r:SUMMARIZED_RELATIONSHIP]-()
    RETURN entities, with_summary, count(DISTINCT r) AS summarized_rels
    """
)[0]
print(f"Entities: {check['entities']} | with summary: {check['with_summary']} "
      f"| SUMMARIZED_RELATIONSHIP edges: {check['summarized_rels']}")
if check["summarized_rels"] == 0:
    print("⚠️  No SUMMARIZED_RELATIONSHIP edges — community reports may be sparse. "
          "Consider setting RUN_NODE_REL_SUMMARIZATION = True.")

## 4. Detect & summarize communities — one library call

This is the core step. `ms_graph.summarize_communities()` does **everything** the
GraphRAG community pipeline needs:

- drops any old GDS projection and creates a fresh one,
- runs **Leiden** to detect the community hierarchy,
- builds the `__Community__` node hierarchy,
- calls the LLM to write a **Query-Focused Summary** report for each community.

The method is **async**, so we `await` it (works directly in a Jupyter cell).
Pass `summarize_all_levels=True` to summarize every hierarchy level instead of just
the top level.

In [ ]:
# Detect + summarize communities using the library.
# summarize_communities() is async -> await it directly in the notebook.
result = await ms_graph.summarize_communities(summarize_all_levels=False)
print(result)   # e.g. "Generated N community summaries"

## 5. Inspect the communities the library created

The `__Community__` nodes and their `summary` reports were written by the library.
Let's read them back.

In [ ]:
# Communities per level
levels = ms_graph.query(
    """
    MATCH (c:__Community__)
    RETURN c.level AS level, count(*) AS communities
    ORDER BY level
    """
)
print("Communities per level:")
for row in levels:
    print(f"  level {row['level']}: {row['communities']} communities")

# Largest communities + their generated summaries
rows = ms_graph.query(
    """
    MATCH (c:__Community__)<-[:IN_COMMUNITY]-(e:__Entity__)
    WITH c, count(e) AS size, collect(e.name)[0..8] AS members
    RETURN c.id AS community, c.level AS level, size, members,
           c.summary AS summary, c.title AS title
    ORDER BY size DESC
    LIMIT 5
    """
)
for r in rows:
    print(f"\n=== Community {r['community']} (level {r['level']}, {r['size']} entities) ===")
    if r.get("title"):
        print("Title  :", r["title"])
    print("Members:", ", ".join(str(m) for m in r["members"]))
    if r.get("summary"):
        summary = r["summary"]
        print("Summary:", summary[:600] + ("..." if len(str(summary)) > 600 else ""))

## 6. Turn the graph into retrieval tools for an agent

Now that we have `__Community__` summaries, we expose **two retrieval functions**
the agent can call:

- **`global_community_search(query)`** — *thematic / "big picture" questions.*
  Map-reduce Query-Focused Summarization: score every community summary against the
  query (map → partial answers), then combine them into one answer (reduce). Great for
  *"What are the main themes / risks / trends?"*

- **`local_entity_search(query)`** — *specific questions about named things.*
  Finds the most relevant entities, pulls their neighbourhood + their community context,
  and answers. Great for *"Tell me about X and how it relates to Y."*

Both return plain text, so they plug straight into any agent framework
(LangChain/LangGraph tools, OpenAI function-calling, etc.).

In [ ]:
# Shared LLM client for the retrieval tools
from openai import OpenAI

llm = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
LLM_MODEL = "gpt-4o"


def _chat(system: str, user: str) -> str:
    resp = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()


def _fetch_community_summaries(level: int | None = None, limit: int = 200):
    """Read the community reports produced by MsGraphRAG."""
    where = "WHERE c.summary IS NOT NULL"
    if level is not None:
        where += f" AND c.level = {level}"
    return ms_graph.query(
        f"""
        MATCH (c:__Community__)
        {where}
        RETURN c.id AS id, c.level AS level, c.title AS title,
               c.summary AS summary, c.rating AS rating
        ORDER BY coalesce(c.rating, 0) DESC
        LIMIT {limit}
        """
    )

In [ ]:
import json


def global_community_search(query: str, level: int | None = None, top_k: int = 8) -> str:
    """
    Answer a THEMATIC / big-picture question using GraphRAG community summaries.

    Map-reduce Query-Focused Summarization:
      1. MAP    - score each community summary for relevance to the query and
                  extract the points that help answer it.
      2. REDUCE - synthesize the useful points into one final answer.

    Args:
        query: The user's high-level question.
        level: Community hierarchy level to use (None = all levels).
        top_k: How many of the most relevant communities to synthesize.
    """
    communities = _fetch_community_summaries(level=level)
    if not communities:
        return "No community summaries found. Run summarize_communities() first."

    # ---- MAP: rate each community's usefulness for this query ----
    scored = []
    for c in communities:
        raw = _chat(
            system=(
                "You extract information from a community report that is useful to "
                "answer the user's question. Respond ONLY as JSON: "
                '{"score": <0-10 integer>, "points": "<key relevant points, or empty>"}.'
            ),
            user=f"Question: {query}\n\nCommunity report:\n{c['title']}\n{c['summary']}",
        )
        try:
            parsed = json.loads(raw)
        except Exception:
            parsed = {"score": 0, "points": ""}
        if parsed.get("score", 0) > 0 and parsed.get("points"):
            scored.append((parsed["score"], c["title"], parsed["points"]))

    if not scored:
        return "No community contained information relevant to the question."

    scored.sort(key=lambda x: x[0], reverse=True)
    top = scored[:top_k]
    mapped = "\n\n".join(f"[{t} | relevance {s}/10]\n{p}" for s, t, p in top)

    # ---- REDUCE: combine into a final answer ----
    return _chat(
        system=(
            "You are an analyst. Using ONLY the provided community findings, write a "
            "comprehensive, well-structured answer to the user's question. Cite the "
            "community titles in [brackets] where relevant. If information is missing, say so."
        ),
        user=f"Question: {query}\n\nCommunity findings:\n{mapped}",
    )


# quick smoke test (uncomment to try)
# print(global_community_search("What are the main themes in this dataset?"))

In [ ]:
def local_entity_search(query: str, top_k: int = 10) -> str:
    """
    Answer a SPECIFIC question about named entities.

    Retrieves the most relevant entities (by simple name/summary match), their
    immediate relationships, and the summary of the community they belong to,
    then asks the LLM to answer using that focused context.

    Args:
        query: A question about specific people/things/concepts.
        top_k: Number of candidate entities to pull context for.
    """
    # Full-text-free candidate match: keyword overlap on name/summary.
    context = ms_graph.query(
        """
        WITH $query AS q
        MATCH (e:__Entity__)
        WHERE toLower(e.name) CONTAINS toLower(q)
           OR any(w IN split(toLower(q), ' ')
                  WHERE size(w) > 3 AND toLower(coalesce(e.summary, e.name)) CONTAINS w)
        WITH e LIMIT $top_k
        OPTIONAL MATCH (e)-[r:SUMMARIZED_RELATIONSHIP]-(nb:__Entity__)
        OPTIONAL MATCH (e)-[:IN_COMMUNITY]->(c:__Community__)
        RETURN e.name AS entity,
               e.summary AS description,
               collect(DISTINCT nb.name)[0..8] AS neighbours,
               collect(DISTINCT c.summary)[0..2] AS community_context
        """,
        params={"query": query, "top_k": top_k},
    )
    if not context:
        return "No matching entities found for this question."

    blocks = []
    for row in context:
        blocks.append(
            f"Entity: {row['entity']}\n"
            f"Description: {row.get('description')}\n"
            f"Related to: {', '.join(row['neighbours']) if row['neighbours'] else 'n/a'}\n"
            f"Community context: {' '.join(cc for cc in row['community_context'] if cc)[:500]}"
        )
    ctx = "\n\n".join(blocks)

    return _chat(
        system=(
            "You answer the user's question using ONLY the provided entity context "
            "(entities, their relationships, and community context). Be specific and "
            "cite entity names. If the context is insufficient, say what is missing."
        ),
        user=f"Question: {query}\n\nContext:\n{ctx}",
    )


# quick smoke test (uncomment to try)
# print(local_entity_search("Tell me about <some entity in your graph>"))

### Give the tools to an agent

Below, the two functions are registered as **OpenAI function-calling tools**. The agent
picks `global_community_search` for thematic questions and `local_entity_search` for
specific ones. (The same functions work as LangChain `@tool`s — just wrap them.)

In [ ]:
# Register the retrieval functions as OpenAI tools
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "global_community_search",
            "description": (
                "Answer THEMATIC / big-picture questions about the whole dataset "
                "(main themes, trends, risks, summaries) using GraphRAG community reports."
            ),
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "The user's question"}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "local_entity_search",
            "description": (
                "Answer SPECIFIC questions about named entities and how they relate "
                "to each other."
            ),
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "The user's question"}},
                "required": ["query"],
            },
        },
    },
]

TOOL_IMPL = {
    "global_community_search": global_community_search,
    "local_entity_search": local_entity_search,
}


def graphrag_agent(question: str, verbose: bool = True) -> str:
    """A minimal agent that decides which GraphRAG tool to call, then answers."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a knowledge-graph analyst. Use global_community_search for "
                "broad/thematic questions and local_entity_search for specific entity "
                "questions. Always call a tool before answering."
            ),
        },
        {"role": "user", "content": question},
    ]

    while True:
        resp = llm.chat.completions.create(
            model=LLM_MODEL, messages=messages, tools=TOOLS, temperature=0
        )
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content

        messages.append(msg)
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            if verbose:
                print(f"🔧 {call.function.name}({args})")
            output = TOOL_IMPL[call.function.name](**args)
            messages.append(
                {"role": "tool", "tool_call_id": call.id, "content": output}
            )


# Example (uncomment to run):
# print(graphrag_agent("What are the overarching themes across the documents?"))
# print(graphrag_agent("Tell me about <some entity> and what it's connected to."))

## 6. Close the connection

Community detection **and** LLM summarization were both performed by the
`ms-graphrag-neo4j` library in section 4 (`summarize_communities()`). Nothing else
is required — the `__Community__` nodes with their QFS summaries now live in your graph.

Finally, close the library connection to release the Neo4j driver.

In [ ]:
# Close the MsGraphRAG connection (also closes the underlying Neo4j driver)
ms_graph.close()
print("🔒 Connection closed.")

## Appendix — Full `ms-graphrag-neo4j` API (reference)

For completeness, this is the **entire library flow**, from raw text to communities.
This notebook uses only the parts relevant to an *existing* graph (init →
optional `summarize_nodes_and_rels` → `summarize_communities` → close). The
`extract_nodes_and_rels` step below is what you'd use to **build** a graph from text.

> Note: the README shows these as synchronous, but in the current library they are
> `async` — use `await` as shown.

In [ ]:
# ======================================================================
# REFERENCE ONLY — full library flow (build a graph from text -> communities)
# Do not run as-is; it would create new nodes in your database.
# ======================================================================
#
# import os
# from ms_graphrag_neo4j import MsGraphRAG
# from neo4j import GraphDatabase
#
# driver = GraphDatabase.driver(
#     os.environ["NEO4J_URI"],
#     auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
# )
# ms_graph = MsGraphRAG(driver=driver, model="gpt-4o")
#
# example_texts = [
#     "Tomaz works for Neo4j",
#     "Tomaz lives in Grosuplje",
#     "Tomaz went to school in Grosuplje",
# ]
# allowed_entities = ["Person", "Organization", "Location"]
#
# # 1) Build the graph from unstructured text
# print(await ms_graph.extract_nodes_and_rels(example_texts, allowed_entities))
#
# # 2) Summarize entities & relationships (creates e.summary + SUMMARIZED_RELATIONSHIP)
# print(await ms_graph.summarize_nodes_and_rels())
#
# # 3) Detect + summarize communities
# print(await ms_graph.summarize_communities())
#
# # 4) Close
# ms_graph.close()